<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/28_tpsl_threshold_single_position_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 28. Take-profit and stop-loss levels, score threshold, single-position simulation

**Questions :** (1) profitability depends on the take-profit / stop-loss levels, which cannot be chosen a priori, so how do precision, recall and profit change with different ATR multipliers? (2) can weighting the training events by their realised profit or loss help? (3) which score threshold should decide whether to open a long or short position? (4) what happens in a realistic simulation where only one position is open at a time, and what would leverage cost?

**Design**
- Pooled event model on 24 tickers (Random Forest, depth 8, balanced classes), same forward split as notebooks 24 to 26: train before 2021-12-15, test from 2022-01-01 (16-day purge).
- Events and features stay the same. Only the triple-barrier labels and the trading exits change with the ATR multipliers, so the labels are recomputed for every pair.
- **Settings are chosen on a validation period (2020-12-15 to 2021-12-01), never on the test period.** The full test grid is shown as a sensitivity analysis, not for picking the best cell.
- Round-trip cost 1.4 bps (IBKR). Maximum holding 10 bars.

In [ ]:
import sys, os
for m in list(sys.modules.keys()):
    if m.startswith('src'):
        del sys.modules[m]

REPO_DIR  = '/content/regime-aware-ml-trading'
PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
if not os.path.isdir(PROJ_ROOT):
    os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
else:
    os.system(f'cd {REPO_DIR} && git pull -q')
os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

import pandas as pd, numpy as np, matplotlib.pyplot as plt
print("src exists:", os.path.isdir(os.path.join(PROJ_ROOT, 'src')))

In [ ]:
from src.experiments.helpers import build_event_data
from src.labeling.label_events import triple_barrier_label
from src.backtest.simulator import simulate_trades
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support, fbeta_score

ALL = ['SPY','QQQ','NVDA','GOOGL','AAPL','MSFT','AMZN','META','AVGO','TSLA','COST','ADBE',
       'AMD','NFLX','INTC','CSCO','JNJ','KO','WMT','PG','CAT','F','XOM','JPM']

event_data = {}
for tk in ALL:
    try:
        event_data[tk] = build_event_data(tk)
    except Exception as e:
        print(tk, "FAILED:", e)

all_cols = sorted(set().union(*[set(d['features'].columns) for d in event_data.values()]))

parts = []
for tk, d in event_data.items():
    lab = d['labeled_df'].reset_index(drop=True)
    assert len(d['features']) == len(lab) == len(d['labels']), tk
    X = d['features'].reindex(columns=all_cols, fill_value=0).fillna(0).reset_index(drop=True)
    meta = pd.DataFrame({'ticker': tk, 'i': np.arange(len(lab)),
                         'date': pd.to_datetime(lab['event_date']).values,
                         'intended': lab['intended_direction'].values})
    parts.append(pd.concat([meta, X], axis=1))
EV = pd.concat(parts, ignore_index=True)
LAB = {tk: d['labeled_df'].reset_index(drop=True) for tk, d in event_data.items()}
RAW = {tk: d['raw'] for tk, d in event_data.items()}
FEATS = all_cols

CUT = pd.Timestamp('2022-01-01'); PURGE = pd.Timedelta(days=16); VAL_START = pd.Timestamp('2020-12-15')
core = (EV.date < VAL_START - PURGE).values                    # training for model selection
val  = ((EV.date >= VAL_START) & (EV.date < CUT - PURGE)).values  # validation (choose settings here)
tr_all = (EV.date < CUT - PURGE).values                        # final training
test = (EV.date >= CUT).values                                 # untouched test period
print("events:", len(EV), "| core / val / test:", core.sum(), val.sum(), test.sum())

In [ ]:
COST = 1.4e-4          # round-trip cost, IBKR 7e-5 per side
MAX_HOLD = 10
_label_cache = {}

def relabel(pt, sl):
    """Triple-barrier labels and realised event return for ATR multipliers (pt, sl).
    Returns (y, ret_pct) aligned with the rows of EV. Features do not change."""
    if (pt, sl) in _label_cache:
        return _label_cache[(pt, sl)]
    y = np.empty(len(EV), dtype=object); r = np.zeros(len(EV))
    for tk, g in EV.groupby('ticker'):
        ev = LAB[tk].set_index('event_date')[['intended_direction']]
        out = triple_barrier_label(RAW[tk], ev, pt_mult=pt, sl_mult=sl, max_holding=MAX_HOLD)
        assert len(out) == len(g), (tk, len(out), len(g))
        y[g.index.values] = out['label'].values
        r[g.index.values] = out['return_pct'].values
    _label_cache[(pt, sl)] = (pd.Series(y, index=EV.index), pd.Series(r, index=EV.index))
    return _label_cache[(pt, sl)]

def fit_rf(mask, y, seed=42, sample_weight=None, n_estimators=300):
    rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=8, class_weight='balanced',
                                random_state=seed, n_jobs=-1)
    return rf.fit(EV.loc[mask, FEATS], y[mask], sample_weight=sample_weight)

def scores(rf, mask):
    """Predicted class, direction (long/short) and trade score (max of p_long, p_short)."""
    P = pd.DataFrame(rf.predict_proba(EV.loc[mask, FEATS]), columns=rf.classes_, index=EV.index[mask])
    direction = P[['long', 'short']].idxmax(axis=1)
    return P, direction, P[['long', 'short']].max(axis=1)

def simulate(mask, pred, pt, sl, cost=COST):
    """Run the trading simulator for every ticker. pred: Series (index = EV index) of long/short/no_trade."""
    out = []
    for tk, g in EV[mask].groupby('ticker'):
        lab_sub = LAB[tk].iloc[g['i'].values]
        tr = simulate_trades(RAW[tk], lab_sub, pred.loc[g.index].values,
                             pt_mult=pt, sl_mult=sl, max_holding=MAX_HOLD, cost_per_trade=cost)
        if len(tr):
            tr['ticker'] = tk; out.append(tr)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

def trade_stats(T):
    if len(T) == 0:
        return dict(trades=0, win_rate=np.nan, mean_net_pct=np.nan, t_stat=np.nan,
                    avg_win_pct=np.nan, avg_loss_pct=np.nan, profit_factor=np.nan)
    r = T.net_return.values; w, l = r[r > 0], r[r <= 0]
    se = r.std(ddof=1) / np.sqrt(len(r)) if len(r) > 1 else np.nan
    return dict(trades=len(r), win_rate=(r > 0).mean(), mean_net_pct=100 * r.mean(),
                t_stat=r.mean() / se if se and se > 0 else np.nan,
                avg_win_pct=100 * w.mean() if len(w) else np.nan,
                avg_loss_pct=100 * l.mean() if len(l) else np.nan,
                profit_factor=w.sum() / abs(l.sum()) if l.sum() != 0 else np.nan)

def clf_stats(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=['long', 'short'],
                                                 average='macro', zero_division=0)
    f05 = fbeta_score(y_true, y_pred, labels=['long', 'short'], beta=0.5, average='macro', zero_division=0)
    return dict(precision=p, recall=r, f05=f05)
print("helpers ready")

## Part A. Take-profit / stop-loss grid

For every pair the model is retrained on labels computed with those multipliers. Precision and recall are for the long and short classes. "No model" trades every event in the detector's own direction.

In [ ]:
PTS = [1.0, 1.5, 2.0, 3.0, 4.0]
SLS = [1.0, 1.5, 2.0, 3.0]

rows = []
for pt in PTS:
    for sl in SLS:
        y, ret = relabel(pt, sl)
        # (1) fit on the core period, score on validation: this is where settings are chosen
        rf_v = fit_rf(core, y, n_estimators=200)
        _, dir_v, sc_v = scores(rf_v, val)
        pred_v = pd.Series(rf_v.predict(EV.loc[val, FEATS]), index=EV.index[val])
        s_val = trade_stats(simulate(val, pred_v, pt, sl))
        # (2) fit on all pre-test data, score on the untouched test period
        rf_t = fit_rf(tr_all, y, n_estimators=200)
        pred_t = pd.Series(rf_t.predict(EV.loc[test, FEATS]), index=EV.index[test])
        s_test = trade_stats(simulate(test, pred_t, pt, sl))
        c_test = clf_stats(y[test], pred_t)
        base = trade_stats(simulate(test, EV.loc[test, 'intended'], pt, sl))   # trade every event, no model
        rows.append(dict(pt=pt, sl=sl, win_label_rate=(y[test] != 'no_trade').mean(),
                         precision=c_test['precision'], recall=c_test['recall'],
                         val_trades=s_val['trades'], val_mean_net_pct=s_val['mean_net_pct'],
                         test_trades=s_test['trades'], win_rate=s_test['win_rate'],
                         mean_net_pct=s_test['mean_net_pct'], t_stat=s_test['t_stat'],
                         profit_factor=s_test['profit_factor'],
                         no_model_mean_net_pct=base['mean_net_pct']))
    print(f"pt={pt} done")

grid = pd.DataFrame(rows)
pd.options.display.float_format = '{:.3f}'.format
print(grid.to_string(index=False))

In [ ]:
# Choose the levels on VALIDATION only (at least 30 validation trades), then read off the test result
cand = grid[grid.val_trades >= 30]
best = cand.loc[cand.val_mean_net_pct.idxmax()]
print("Levels chosen on validation: pt =", best.pt, " sl =", best.sl)
print(best[['val_trades', 'val_mean_net_pct', 'test_trades', 'precision', 'recall', 'win_rate',
            'mean_net_pct', 't_stat', 'profit_factor', 'no_model_mean_net_pct']].to_string())
PT_BEST, SL_BEST = float(best.pt), float(best.sl)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, col, title in zip(axes, ['mean_net_pct', 'precision', 'no_model_mean_net_pct'],
        ['Pooled event model: mean net % per trade (test)', 'Precision on long/short (test)',
         'No model, all events: mean net % per trade (test)']):
    pv = grid.pivot(index='sl', columns='pt', values=col)
    im = ax.imshow(pv.values, cmap='RdYlGn', aspect='auto', origin='lower',
                   vmin=-abs(pv.values).max() if col != 'precision' else None,
                   vmax=abs(pv.values).max() if col != 'precision' else None)
    ax.set_xticks(range(len(pv.columns))); ax.set_xticklabels(pv.columns)
    ax.set_yticks(range(len(pv.index))); ax.set_yticklabels(pv.index)
    for (i, j), v in np.ndenumerate(pv.values):
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8)
    ax.set_xlabel('take-profit (ATR multiples)'); ax.set_ylabel('stop-loss (ATR multiples)'); ax.set_title(title, fontsize=9)
plt.tight_layout(); plt.show()

## Part B. Training events weighted by realised profit or loss

Each training event gets a weight equal to the size of its realised move (the absolute return of its intended trade, capped at 5 times the average), so the forest pays more attention to the rare large moves. Mean over 3 seeds.

In [ ]:
def weighted_vs_plain(pt, sl, seeds=(42, 7, 123)):
    y, ret = relabel(pt, sl)
    w = ret.abs()
    w = (w / w[tr_all].mean()).clip(upper=5.0)          # event weight = size of its realised move
    out = []
    for name, sw in [('plain', None), ('profit-weighted', w[tr_all].values)]:
        for s in seeds:
            rf = fit_rf(tr_all, y, seed=s, sample_weight=sw)
            pred = pd.Series(rf.predict(EV.loc[test, FEATS]), index=EV.index[test])
            st = trade_stats(simulate(test, pred, pt, sl)); cl = clf_stats(y[test], pred)
            out.append(dict(pt=pt, sl=sl, training=name, seed=s, **cl, **st))
    o = pd.DataFrame(out)
    return o.groupby(['pt', 'sl', 'training'])[['precision', 'recall', 'trades', 'win_rate',
                                                 'mean_net_pct', 't_stat', 'profit_factor']].mean()

print(pd.concat([weighted_vs_plain(2.0, 2.0), weighted_vs_plain(PT_BEST, SL_BEST)]).drop_duplicates().to_string())

## Part C. Which score threshold opens a position?

The model gives a score for long and for short. A position opens only if the larger of the two is at least the threshold, in the direction of the larger score. The threshold is chosen on the validation period in two ways (best F0.5, best mean net return) and then read off on the test period.

In [ ]:
THR = np.round(np.arange(0.25, 0.71, 0.05), 2)

def threshold_table(pt, sl, mask, rf):
    y, _ = relabel(pt, sl)
    P, direction, score = scores(rf, mask)
    rows = []
    for t in THR:
        pred = pd.Series(np.where(score >= t, direction, 'no_trade'), index=P.index)
        st = trade_stats(simulate(mask, pred, pt, sl)); cl = clf_stats(y[mask], pred)
        rows.append(dict(threshold=t, **cl, **st))
    return pd.DataFrame(rows)

def threshold_study(pt, sl):
    y, _ = relabel(pt, sl)
    tv = threshold_table(pt, sl, val, fit_rf(core, y))
    tt = threshold_table(pt, sl, test, fit_rf(tr_all, y))
    ok = tv[tv.trades >= 30]
    t_f05 = ok.loc[ok.f05.idxmax(), 'threshold']                 # criterion 1: F0.5 on validation
    t_net = ok.loc[ok.mean_net_pct.idxmax(), 'threshold']        # criterion 2: mean net return on validation
    print(f"pt={pt}, sl={sl}: threshold chosen on validation  F0.5 -> {t_f05},  mean net -> {t_net}")
    print("\nVALIDATION"); print(tv[['threshold','precision','recall','f05','trades','mean_net_pct']].to_string(index=False))
    print("\nTEST");       print(tt[['threshold','precision','recall','f05','trades','win_rate','mean_net_pct','t_stat']].to_string(index=False))
    return tv, tt, t_f05, t_net

tv22, tt22, t_f05_22, t_net_22 = threshold_study(2.0, 2.0)

In [ ]:
tvB, ttB, t_f05_B, t_net_B = threshold_study(PT_BEST, SL_BEST)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (tt, name) in zip(axes, [(tt22, 'pt=2, sl=2'), (ttB, f'pt={PT_BEST}, sl={SL_BEST} (chosen on validation)')]):
    ax.plot(tt.threshold, tt.mean_net_pct, marker='o', color='#4C72B0', label='mean net % per trade')
    ax.axhline(0, color='k', lw=0.8); ax.set_xlabel('score threshold'); ax.set_ylabel('mean net % per trade (test)')
    ax2 = ax.twinx(); ax2.plot(tt.threshold, tt.trades, color='#8C8C8C', ls='--', label='number of trades'); ax2.set_ylabel('trades')
    ax.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

## Part D. One position at a time across all 24 tickers (2022 to 2025)

Whenever the account is flat, the model signal with the largest score among all tickers is taken; while a trade is open every other signal is ignored. Trades use the full account, with compounding, 1.4 bps cost, no leverage.

In [ ]:
def portfolio(pt, sl, threshold=0.0, leverage=1.0, fin_rate=0.0, seed=42, verbose=True):
    """One open position at a time across ALL tickers. If several tickers signal on the same day,
    take the one with the largest score. While a trade is open, other signals are ignored."""
    y, _ = relabel(pt, sl)
    rf = fit_rf(tr_all, y, seed=seed)
    P, direction, score = scores(rf, test)
    pred = pd.Series(np.where(score >= threshold, direction, 'no_trade'), index=P.index)
    cands = []
    for idx in pred.index[pred != 'no_trade']:
        tk, i = EV.at[idx, 'ticker'], EV.at[idx, 'i']
        tr = simulate_trades(RAW[tk], LAB[tk].iloc[[i]], [pred.loc[idx]], pt_mult=pt, sl_mult=sl,
                             max_holding=MAX_HOLD, cost_per_trade=COST)
        if len(tr):
            t = tr.iloc[0]
            cands.append(dict(ticker=tk, entry=pd.Timestamp(t.event_date), exit=pd.Timestamp(t.exit_date),
                              net=t.net_return, score=score.loc[idx], direction=t.direction))
    C_ = pd.DataFrame(cands).sort_values(['entry', 'score'], ascending=[True, False])
    taken, free_after = [], pd.Timestamp.min
    for day, g in C_.groupby('entry'):
        if day <= free_after:
            continue                                  # still in a position
        r = g.iloc[0]                                 # largest score that day
        taken.append(r); free_after = r['exit']
    T = pd.DataFrame(taken).reset_index(drop=True)
    days = (T['exit'] - T['entry']).dt.days.clip(lower=1)
    T['net_lev'] = leverage * T['net'] - (leverage - 1) * fin_rate * days / 365    # financing on the borrowed part
    eq = (1 + T['net_lev']).cumprod()
    span = (T['exit'].max() - pd.Timestamp('2022-01-01')).days / 365.25
    dd = (eq / eq.cummax() - 1).min()
    res = dict(signals=len(C_), trades=len(T), win_rate=(T.net > 0).mean(), mean_net_pct=100 * T.net.mean(),
               total_return_pct=100 * (eq.iloc[-1] - 1), cagr_pct=100 * (eq.iloc[-1] ** (1 / span) - 1),
               max_drawdown_pct=100 * dd, time_in_market=days.sum() / ((T['exit'].max() - pd.Timestamp('2022-01-01')).days))
    return res, T, eq

res1, T1, eq1 = portfolio(2.0, 2.0)
res2, T2, eq2 = portfolio(PT_BEST, SL_BEST)
spy = RAW['SPY']['Close']; spy = spy[spy.index >= CUT]
bh = 100 * (spy.iloc[-1] / spy.iloc[0] - 1)
print(pd.DataFrame({'pt2_sl2': res1, f'pt{PT_BEST}_sl{SL_BEST}': res2}).to_string())
print(f"\nSPY buy-and-hold over the same period: {bh:.1f}%")

plt.figure(figsize=(9, 3.8))
plt.plot(T1['exit'], eq1.values, label='pooled event model, pt=2 sl=2, one position at a time')
plt.plot(T2['exit'], eq2.values, label=f'levels chosen on validation (pt={PT_BEST}, sl={SL_BEST})')
plt.plot(spy.index, spy.values / spy.iloc[0], color='grey', lw=1, label='SPY buy and hold')
plt.axhline(1, color='k', lw=0.6); plt.ylabel('equity (start = 1)'); plt.legend(fontsize=8); plt.title('Single-position simulation, 2022 to 2025')
plt.tight_layout(); plt.show()

## Part E. Leverage

Leverage multiplies each trade's return. The borrowed part (leverage minus 1) pays a financing rate for the days the trade is open. The 6% annual rate is an assumption for illustration; the real broker rate has to be checked.

In [ ]:
FIN_RATE = 0.06      # ASSUMED annual borrowing rate; check the current IBKR margin rate before quoting it
rows = []
for L in [1, 2, 3]:
    for fr in [0.0, FIN_RATE]:
        r, _, _ = portfolio(PT_BEST, SL_BEST, leverage=L, fin_rate=fr)
        rows.append(dict(leverage=L, financing_rate=fr, trades=r['trades'], total_return_pct=r['total_return_pct'],
                         cagr_pct=r['cagr_pct'], max_drawdown_pct=r['max_drawdown_pct']))
print(pd.DataFrame(rows).to_string(index=False))